# Automated Atrial Fibrillation (AF) Detection via DCCA Feature Fusion
This notebook runs the complete 5-fold cross-validation pipeline for Atrial Fibrillation detection from single-lead ECG signals using the PhysioNet/CinC Challenge 2017 dataset.

### Pipeline Steps:
1. **Denoising**: Butterworth bandpass filter (0.5 - 45 Hz).
2. **Deep Features**: 1D-ResNet with 6 residual blocks followed by a Gated Recurrent Unit (GRU).
3. **Traditional Features**: R-peak detection, RR-interval stats, P-wave analysis, and Welch PSD power bands.
4. **Feature Fusion**: Discriminant Canonical Correlation Analysis (DCCA) to optimize intra-class correlation and minimize inter-class correlation.
5. **Classification**: Fully Connected Neural Network.

## 1. Setup Environment
Make sure Internet is **Enabled** in the Kaggle sidebar settings to download requirements and the PhysioNet dataset.

In [ ]:
# Install requirements
!pip install -r requirements.txt

## 2. Run Pipeline Test
Before running on the entire dataset, let's verify everything works using the pipeline validation test with a mock dataset.

In [ ]:
!python test_pipeline.py

## 3. Run Full 5-Fold Cross-Validation
Now we run the complete pipeline on the PhysioNet Challenge 2017 dataset. 
The dataset zip file (approx. 95MB) will be automatically downloaded and extracted into the `./data` directory.

In [ ]:
!python src/train.py --data_root ./data --folds 5 --deep_epochs 20 --classifier_epochs 40

## 4. Run Custom Inference
Here is how you can use the trained components to run inference on a new raw ECG signal.

In [ ]:
import numpy as np
import torch
from src.dataset import PhysioNet2017Dataset
from src.features import extract_features

def run_single_inference(ecg_signal_raw, fs=300):
    # 1. Preprocess & filter raw ECG signal
    filtered_signal = PhysioNet2017Dataset.butter_bandpass_filter(ecg_signal_raw, lowcut=0.5, highcut=45.0, fs=fs)
    # Standardize
    filtered_signal = (filtered_signal - np.mean(filtered_signal)) / (np.std(filtered_signal) + 1e-8)
    
    # 2. Extract handcrafted features
    x_handcrafted = extract_features(filtered_signal, fs=fs)
    
    print("Handcrafted feature vector shape:", x_handcrafted.shape)
    # Note: For actual inference, you would project these using the fitted DCCA class and
    # feed them to the trained FusionClassifier. 
    
run_single_inference(np.random.normal(0, 1, 9000))